# TinyRAG Colab CUDA Demo

This notebook clones the TinyRAG repository, installs the CUDA llama.cpp runtime with `uv`, downloads the configured GGUF model, rebuilds the local RAG index, and runs grounded generation with GPU offload.

Before running all cells, set **Runtime > Change runtime type > GPU** in Colab.

## Configuration

Change `MODEL_REPO`, `MODEL_FILE`, and the runtime settings here to try a different GGUF model. Keep `N_CTX` and `N_GPU_LAYERS` conservative until VRAM usage is confirmed.

In [ ]:
REPO_URL = "https://github.com/cxyfer/tinyrag-laptop-assistant.git"
BRANCH = "main"
PROJECT_DIR = "/content/tinyrag-laptop-assistant"

MODEL_REPO = "Qwen/Qwen2.5-1.5B-Instruct-GGUF"
MODEL_FILE = "qwen2.5-1.5b-instruct-q4_k_m.gguf"
MODEL_PATH = f"models/{MODEL_FILE}"

QUESTION = "BXH 使用哪一張顯示卡？"
N_CTX = 2048
TEMPERATURE = 0.1
MAX_TOKENS = 256
N_GPU_LAYERS = 35

## Check the GPU runtime

In [ ]:
!nvidia-smi

## Clone the repository

In [ ]:
!rm -rf {PROJECT_DIR}
!git clone --branch {BRANCH} {REPO_URL} {PROJECT_DIR}
%cd {PROJECT_DIR}

## Install the CUDA llama.cpp runtime

In [ ]:
!python -m pip install -q uv
!uv sync --frozen --extra llama-cu121

In [ ]:
!uv run --frozen --extra llama-cu121 python - <<'PY'
from llama_cpp import llama_cpp

print("gpu_offload_supported=", llama_cpp.llama_supports_gpu_offload())
PY

## Download the GGUF model

In [ ]:
!uvx --from huggingface-hub hf download {MODEL_REPO} {MODEL_FILE} --local-dir models

## Build the local RAG index

In [ ]:
!uv run --frozen --extra llama-cu121 tinyrag ingest --prefer-cache
!uv run --frozen --extra llama-cu121 tinyrag build-index

## Ask a grounded question

In [ ]:
!uv run --frozen --extra llama-cu121 tinyrag ask "{QUESTION}" --model-path {MODEL_PATH} --n-ctx {N_CTX} --temperature {TEMPERATURE} --max-tokens {MAX_TOKENS} --n-gpu-layers {N_GPU_LAYERS}

## Optional: run the real-model benchmark

In [ ]:
!uv run --frozen --extra llama-cu121 tinyrag benchmark --use-model --model-path {MODEL_PATH} --n-ctx {N_CTX} --temperature {TEMPERATURE} --max-tokens {MAX_TOKENS} --n-gpu-layers {N_GPU_LAYERS}